In [4]:
import sys
from pathlib import Path


def repo_root() -> Path:
    """Folder that contains ``src/`` (works from repo root, ``notebooks/``, or IDE cwd)."""
    cur = Path.cwd().resolve()
    for d in [cur, *cur.parents]:
        if (d / "src" / "Feature_Extraction.py").is_file():
            return d
    raise RuntimeError("Could not find project root (missing src/Feature_Extraction.py).")


PROJECT_ROOT = repo_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from sklearn.model_selection import train_test_split

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\pc\Desktop\Projects\Natural-Lang-Processing-Arabic\arabic-sentiment-analysis


In [5]:
data = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "tweets_clean.csv")
data = data.dropna(subset=["clean_text", "label"]).reset_index(drop=True)

print("Shape:", data.shape)
print("Columns:", list(data.columns))
data.head()

Shape: (10001, 2)
Columns: ['clean_text', 'label']


,clean_text,label
0,بعد استقال رئيس محكم دستوري ننتظر استقال رئيس ...,OBJ
1,اهنئ دكتور احمد جمال دين قياد حزب مصر مناسب صد...,POS
2,برادع يستقو امريك مرهاخر و يرسل عصام عري الي ا...,NEG
3,حري عدال شاهد الان يله اتحادي اول يلم استقصائ ...,OBJ
4,والد لو اقول خاطر حشيش تضح بس من اقول ملل الل ...,NEUTRAL


In [6]:
from src.Feature_Extraction import vectorize

X_train, X_test, y_train, y_test = train_test_split(
    data["clean_text"].astype(str),
    data["label"],
    test_size=0.2,
    random_state=42,
    stratify=data["label"],
)

# Same as tfidf(...); use method="bow" for bag-of-words
X_train_tfidf, X_test_tfidf = vectorize(X_train, X_test, method="tfidf")

print("Train matrix:", X_train_tfidf.shape)
print("Test matrix: ", X_test_tfidf.shape)

Train matrix: (8000, 5000)
Test matrix:  (2001, 5000)


In [7]:
from src.Feature_Selection import select_features

# Defaults: method="chi2", k=1000 — override with method="mi" or method="fisher"
X_train_sel, X_test_sel = select_features(X_train_tfidf, y_train, X_test_tfidf)


In [8]:
from src.models import train_classifier, predict_labels
from src.evaluation import evaluate_predictions, build_classification_report

clf = train_classifier("logistic", X_train_sel, y_train)
y_pred = predict_labels(clf, X_test_sel)

print(evaluate_predictions(y_test, y_pred))
print(build_classification_report(y_test, y_pred))

{'accuracy': 0.6876561719140429, 'f1_macro': 0.3286870114040879}
              precision    recall  f1-score   support

         NEG       0.50      0.19      0.28       337
     NEUTRAL       0.44      0.05      0.09       166
         OBJ       0.70      0.96      0.81      1338
         POS       0.71      0.07      0.14       160

    accuracy                           0.69      2001
   macro avg       0.59      0.32      0.33      2001
weighted avg       0.65      0.69      0.61      2001

